In [148]:
import pandas as pd
import plotly.express as px
import folium
import yaml
import numpy as np
import geopandas as gpd
from pyproj import Transformer
from shapely.geometry import Polygon
from folium import plugins
import requests

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=False)

In [109]:
# ============================================================================
# CONFIGURATION
# ============================================================================
mode = 'plot'  # Set to 'plot' to generate map, or 'data' to skip plotting

# Define neighborhood polygon corners
neighborhood_corners = [
    (52.340217730809734, 4.952094954784996),   # Corner 1
    (52.35568485554889, 4.926603240307636),   # Corner 2
    (52.35888246717443, 4.940593642529217),  # Corner 3
    (52.35835828440313, 4.959304732616976),
    (52.35642858299969, 4.967018765743752),  # Corner 4
]

# Create polygon from corners (convert lat,lon to lon,lat for Shapely)
polygon_coords = [(lon, lat) for lat, lon in neighborhood_corners]
neighborhood_polygon = Polygon(polygon_coords)

# Get bounding box from polygon
bounds = neighborhood_polygon.bounds  # (minx, miny, maxx, maxy)
bbox_1098 = {
    'minx': bounds[0],
    'miny': bounds[1],
    'maxx': bounds[2],
    'maxy': bounds[3]
}

#print("=" * 80)
#print("NEIGHBORHOOD POLYGON CONFIGURATION")
#print("=" * 80)
#print(f"Corners (lat, lon):")
#for i, (lat, lon) in enumerate(neighborhood_corners, 1):
#    print(f"  {i}. ({lat}, {lon})")
#print(f"\nBounding box: {bbox_1098}")
#print()

# ============================================================================
# QUERY LIANDER LAYERS
# ============================================================================
base_service_url = "https://services1.arcgis.com/v6W5HAVrpgSg3vts/arcgis/rest/services/Liander_Open_Data_Elektra/FeatureServer"

layers_to_query = {
    424: "Middenspanningskabel (MV cables)",
    641: "Hoogspanningsstation (HV stations)"
}

#print("=" * 80)
#print("QUERYING MV CABLES & HV STATIONS")
#print("=" * 80)
#print()

# Transform bbox to RD
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
x_min_rd, y_min_rd = transformer.transform(bbox_1098['minx'], bbox_1098['miny'])
x_max_rd, y_max_rd = transformer.transform(bbox_1098['maxx'], bbox_1098['maxy'])

layers_data_1098 = {}

for layer_id, layer_name in layers_to_query.items():
    layer_url = f"{base_service_url}/{layer_id}/query"
    
    params = {
        'geometry': f'{x_min_rd},{y_min_rd},{x_max_rd},{y_max_rd}',
        'geometryType': 'esriGeometryEnvelope',
        'inSR': '28992',
        'outSR': '4326',
        'spatialRel': 'esriSpatialRelIntersects',
        'outFields': '*',
        'returnGeometry': 'true',
        'f': 'geojson',
        'resultRecordCount': 5000
    }
    
    try:
        #print(f"[{layer_id}] Querying {layer_name}...")
        response = requests.get(layer_url, params=params, timeout=15)
        response.raise_for_status()
        
        geojson = response.json()
        feature_count = len(geojson.get('features', []))
        
        if feature_count > 0:
            gdf = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
            # Filter to only features within the polygon
            gdf = gdf[gdf.geometry.within(neighborhood_polygon)]
            layers_data_1098[layer_id] = gdf
            #print(f"  ✓ Retrieved {feature_count} features (filtered to {len(gdf)} within polygon)")
            #print(f"    Geometry types: {gdf.geometry.type.unique()}")
        else:
            print(f"  ✗ No features found")
            
    except Exception as e:
        print(f"  ✗ Error: {e}")

# ============================================================================
# NETWORK STATISTICS
# ============================================================================
#print("=" * 80)
#print("NETWORK STATISTICS")
#print("=" * 80)
for layer_id, gdf in layers_data_1098.items():
    layer_name = layers_to_query[layer_id]
    
    if layer_id == 641:
        pass
    else:
        # Project to EPSG:28992 (Dutch RD) for accurate length calculation
        gdf_projected = gdf.to_crs('EPSG:28992')
        total_length_m = gdf_projected.geometry.length.sum()


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# PLOTTING FUNCTION (REUSABLE FOR RAW & CLEANED NETWORKS)
# ============================================================================

def plot_network_topology(
    gdf_mv_cables,
    gdf_hv_stations,
    title="Network Topology",
    output_file=None,
    bbox=None,
    polygon=None,
    zoom_start=15,
    show_layer_control=True
):
    """
    Plot MV cables and HV stations on an interactive folium map.
    
    Compatible with both raw ArcGIS data and cleaned/processed networks.
    
    Parameters:
    -----------
    gdf_mv_cables : GeoDataFrame
        LineString geometries of MV cables (can be raw or cleaned)
    gdf_hv_stations : GeoDataFrame
        Point geometries of HV stations (can be raw or cleaned)
    title : str
        Map title to display
    output_file : str, optional
        Path to save HTML map. If None, only returns map object.
    bbox : dict, optional
        Bounding box with keys {'minx', 'miny', 'maxx', 'maxy'}.
        If None, computed from GeoDataFrames.
    polygon : Polygon, optional
        Shapely Polygon to plot as neighborhood boundary (dashed line).
    zoom_start : int
        Initial zoom level (default: 15)
    show_layer_control : bool
        Add layer control widget to map (default: True)
    
    Returns:
    --------
    folium.Map
        The folium map object (for further customization if needed)
    """
    
    # ====================================================================
    # Compute map center and bounds
    # ====================================================================
    if bbox is None:
        # Auto-compute from GeoDataFrames
        all_geoms = pd.concat([
            gdf_mv_cables.geometry if len(gdf_mv_cables) > 0 else gpd.GeoSeries(),
            gdf_hv_stations.geometry if len(gdf_hv_stations) > 0 else gpd.GeoSeries()
        ])
        if len(all_geoms) > 0:
            bounds = all_geoms.total_bounds  # (minx, miny, maxx, maxy)
            bbox = {
                'minx': bounds[0],
                'miny': bounds[1],
                'maxx': bounds[2],
                'maxy': bounds[3]
            }
        else:
            # Fallback: default Amsterdam center
            bbox = {'minx': 4.9, 'miny': 52.35, 'maxx': 4.95, 'maxy': 52.36}
    
    center_lat = (bbox['miny'] + bbox['maxy']) / 2
    center_lon = (bbox['minx'] + bbox['maxx']) / 2
    
    # ====================================================================
    # Create map
    # ====================================================================
    map_obj = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=zoom_start,
        tiles="CartoDB positron"
    )
    
    # ====================================================================
    # Add neighborhood polygon boundary (if provided)
    # ====================================================================
    if polygon is not None:
        poly_coords = [(lat, lon) for lon, lat in polygon.exterior.coords]
        folium.PolyLine(
            locations=poly_coords,
            color='#333333',
            weight=3,
            opacity=0.8,
            dash_array='5, 5',
            popup='Neighborhood Boundary'
        ).add_to(map_obj)
    
    # ====================================================================
    # Plot HV Stations (Points)
    # ====================================================================
    if len(gdf_hv_stations) > 0:
        print(f"Plotting {len(gdf_hv_stations)} HV stations...")
        
        for idx, row in gdf_hv_stations.iterrows():
            if row.geometry.geom_type == 'Point':
                # Build popup text from available columns
                popup_text = "<b>HV Station</b><br>"
                for col in gdf_hv_stations.columns:
                    if col != 'geometry' and pd.notna(row[col]):
                        value = row[col]
                        # Format numbers nicely
                        if isinstance(value, (int, float)):
                            popup_text += f"{col}: {value:.2f}<br>"
                        else:
                            popup_text += f"{col}: {value}<br>"
                
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=8,
                    popup=folium.Popup(popup_text, max_width=250),
                    color='#FF6600',           # Orange
                    fill=True,
                    fillColor='#FF6600',
                    fillOpacity=0.8,
                    weight=2,
                    tooltip=f"HV Station"
                ).add_to(map_obj)
    
    # ====================================================================
    # Plot MV Cables (LineStrings)
    # ====================================================================
    if len(gdf_mv_cables) > 0:
        print(f"Plotting {len(gdf_mv_cables)} MV cable segments...")
        
        feature_group = folium.FeatureGroup(name="MV Cables (Layer 424)")
        
        for idx, row in gdf_mv_cables.iterrows():
            if row.geometry.geom_type == 'LineString':
                coords = [(point[1], point[0]) for point in row.geometry.coords]
                
                # Build popup text from available columns
                popup_text = "<b>MV Cable</b><br>"
                for col in gdf_mv_cables.columns[:5]:  # Limit to first 5 columns
                    if col != 'geometry' and pd.notna(row[col]):
                        value = row[col]
                        # Format numbers nicely
                        if isinstance(value, (int, float)):
                            popup_text += f"{col}: {value:.2f}<br>"
                        else:
                            popup_text += f"{col}: {value}<br>"
                
                folium.PolyLine(
                    locations=coords,
                    color='#0066FF',           # Blue
                    weight=2,
                    opacity=0.7,
                    popup=folium.Popup(popup_text, max_width=250),
                    tooltip="MV Cable Segment"
                ).add_to(feature_group)
        
        feature_group.add_to(map_obj)
    
    # ====================================================================
    # Add title, layer control, and styling
    # ====================================================================
    if show_layer_control:
        folium.LayerControl().add_to(map_obj)
    
    title_html = f'<h3 align="center" style="font-size:20px"><b>{title}</b></h3>'
    map_obj.get_root().html.add_child(folium.Element(title_html))
    
    # ====================================================================
    # Save to file if specified
    # ====================================================================
    if output_file is not None:
        map_obj.save(output_file)
        print(f"✓ Map saved to: {output_file}")
    
    return map_obj


# ============================================================================
# PLOT RAW NETWORK (Original ArcGIS data)
# ============================================================================

print("=" * 80)
print("PLOTTING RAW NETWORK (Before Cleaning)")
print("=" * 80)
print()

if 424 in layers_data_1098 and 641 in layers_data_1098:
    map_raw = plot_network_topology(
        gdf_mv_cables=layers_data_1098[424],
        gdf_hv_stations=layers_data_1098[641],
        title="Postal Code 1098 - MV & HV Network (Raw ArcGIS Data)",
        output_file="liander_network_1098_raw.html",
        bbox=bbox_1098,
        polygon=neighborhood_polygon,
        zoom_start=15
    )
    print()
else:
    print("⚠ Cannot plot raw network - missing layer data")
    print()

PLOTTING RAW NETWORK (Before Cleaning)

Plotting 1 HV stations...
Plotting 993 MV cable segments...
✓ Map saved to: liander_network_1098_raw.html



In [ ]:
# ============================================================================
# STAGE 1: DATA CLEANING & VALIDATION (DISTANCE-BASED MERGE → SNAP)
# ============================================================================

print("=" * 80)
print("STAGE 1: DATA CLEANING & VALIDATION (DISTANCE-BASED MERGE → SNAP)")
print("=" * 80)
print()

from sklearn.cluster import DBSCAN
from shapely.geometry import LineString, Point
import time


def merge_nearby_cables_fast(gdf, proximity_threshold_m=50, direction_tolerance_degrees=15):
    """
    Merge nearby parallel cables using distance-based clustering (fast).
    Works on full LineString geometries without explosion.
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Input cables with LineString geometries (in EPSG:4326)
    proximity_threshold_m : float
        Distance threshold (meters) for considering cables as nearby
    direction_tolerance_degrees : float
        Maximum angle difference (degrees) for cables to be considered parallel
    
    Returns:
    --------
    GeoDataFrame
        Merged cable trunks (in EPSG:4326)
    """
    print(f"  Merging nearby parallel cables using distance-based clustering...")
    print(f"    Proximity threshold: {proximity_threshold_m}m")
    print(f"    Direction tolerance: {direction_tolerance_degrees}°")
    
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.geom_type == 'LineString'].copy()
    gdf = gdf.reset_index(drop=True)
    
    if len(gdf) == 0:
        print(f"  No LineStrings to merge")
        return gdf
    
    gdf_proj = gdf.to_crs('EPSG:28992')
    
    # Calculate cable properties (in projected CRS for accurate calculations)
    gdf_proj['length'] = gdf_proj.geometry.length
    gdf_proj['start_pt'] = gdf_proj.geometry.apply(lambda g: np.array(g.coords[0]))
    gdf_proj['end_pt'] = gdf_proj.geometry.apply(lambda g: np.array(g.coords[-1]))
    gdf_proj['midpoint'] = gdf_proj.apply(
        lambda row: (row['start_pt'] + row['end_pt']) / 2,
        axis=1
    )
    gdf_proj['angle'] = gdf_proj.apply(
        lambda row: np.arctan2(
            row['end_pt'][1] - row['start_pt'][1],
            row['end_pt'][0] - row['start_pt'][0]
        ) * 180 / np.pi,
        axis=1
    )
    
    # Use DBSCAN to cluster cable midpoints by proximity
    midpoints = np.array([pt for pt in gdf_proj['midpoint']])
    dbscan = DBSCAN(eps=proximity_threshold_m, min_samples=1).fit(midpoints)
    proximity_clusters = dbscan.labels_
    
    # Group cables by proximity cluster
    merged_lines = []
    cluster_properties = []
    
    # Within each proximity cluster, group by direction similarity
    for prox_cluster_id in set(proximity_clusters):
        prox_mask = proximity_clusters == prox_cluster_id
        prox_indices = np.where(prox_mask)[0]
        
        if len(prox_indices) == 0:
            continue
        
        # Get angles for cables in this proximity cluster
        angles = gdf_proj.iloc[prox_indices]['angle'].values
        
        # Sub-cluster by direction similarity
        direction_clusters = []
        used = set()
        
        for i, idx in enumerate(prox_indices):
            if i in used:
                continue
            
            base_angle = angles[i]
            dir_cluster = [idx]
            used.add(i)
            
            for j, other_idx in enumerate(prox_indices):
                if j in used:
                    continue
                
                angle_diff = abs(base_angle - angles[j])
                if angle_diff > 90:
                    angle_diff = 180 - angle_diff
                
                if angle_diff <= direction_tolerance_degrees:
                    dir_cluster.append(other_idx)
                    used.add(j)
            
            direction_clusters.append(dir_cluster)
        
        # Create merged line for each direction cluster
        for dir_cluster in direction_clusters:
            if len(dir_cluster) == 0:
                continue
            
            cluster = gdf_proj.iloc[dir_cluster]
            
            # Average start/end points of all cables in cluster
            all_start_pts = np.array([pt for pt in cluster['start_pt']])
            all_end_pts = np.array([pt for pt in cluster['end_pt']])
            
            avg_start = np.mean(all_start_pts, axis=0)
            avg_end = np.mean(all_end_pts, axis=0)
            
            # Create merged line
            merged_geom = LineString([avg_start, avg_end])
            if merged_geom.is_valid:
                merged_lines.append(merged_geom)
                cluster_properties.append({
                    'num_cables': len(dir_cluster),
                    'first_row': cluster.iloc[0]
                })
    
    # Create output GeoDataFrame
    gdf_merged = gpd.GeoDataFrame(
        geometry=merged_lines,
        crs='EPSG:28992'
    )
    
    # Preserve properties from first cable in each cluster
    for idx, prop in enumerate(cluster_properties):
        if idx < len(gdf_merged):
            first_row = prop['first_row']
            for col in gdf.columns:
                if col != 'geometry' and col not in ['length', 'start_pt', 'end_pt', 'angle', 'midpoint']:
                    if col in first_row.index:
                        gdf_merged.loc[idx, col] = first_row[col]
    
    # Convert back to WGS84
    gdf_merged = gdf_merged.to_crs('EPSG:4326')
    
    num_merged = len(gdf) - len(gdf_merged)
    print(f"  ✓ Merged {len(gdf)} cables → {len(gdf_merged)} trunk(s) ({num_merged} cables merged)")
    
    return gdf_merged


def snap_endpoints_simple(gdf, snap_distance_m=15):
    """
    Lightweight endpoint snapping using DBSCAN clustering.
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Input cables with LineString geometries (in EPSG:4326)
    snap_distance_m : float
        Maximum distance (meters) to merge endpoints
    
    Returns:
    --------
    GeoDataFrame
        Snapped cables (in EPSG:4326)
    """
    print(f"  Snapping endpoints (tolerance: {snap_distance_m}m)...")
    
    gdf = gdf.copy()
    gdf_proj = gdf.to_crs('EPSG:28992')
    
    # Extract all endpoints
    endpoints = []
    endpoint_to_geom = {}
    geom_idx = 0
    
    for idx, geom in gdf_proj.geometry.items():
        if geom.geom_type == 'LineString':
            coords = list(geom.coords)
            if len(coords) >= 2:
                start_pt = (coords[0][0], coords[0][1])
                end_pt = (coords[-1][0], coords[-1][1])
                
                endpoints.append(start_pt)
                endpoints.append(end_pt)
                endpoint_to_geom[len(endpoints) - 2] = (idx, 'start')
                endpoint_to_geom[len(endpoints) - 1] = (idx, 'end')
        
        geom_idx += 1
    
    if len(endpoints) == 0:
        return gdf
    
    endpoints = np.array(endpoints)
    
    # Cluster endpoints using DBSCAN
    clustering = DBSCAN(eps=snap_distance_m, min_samples=1).fit(endpoints)
    cluster_labels = clustering.labels_
    
    # Build snap mapping
    snap_mapping = {}
    for cluster_id in set(cluster_labels):
        cluster_mask = cluster_labels == cluster_id
        cluster_points = endpoints[cluster_mask]
        centroid = np.mean(cluster_points, axis=0)
        
        for point in cluster_points:
            snap_mapping[tuple(point)] = tuple(centroid)
    
    # Apply snapping
    snapped_geoms = []
    
    for geom in gdf_proj.geometry:
        if geom.geom_type == 'LineString':
            coords = list(geom.coords)
            snapped_coords = []
            
            for i, coord in enumerate(coords):
                coord_tuple = (coord[0], coord[1])
                # Only snap endpoints, not interior points
                if i == 0 or i == len(coords) - 1:
                    snapped_coord = snap_mapping.get(coord_tuple, coord_tuple)
                    snapped_coords.append(snapped_coord)
                else:
                    snapped_coords.append(coord)
            
            # Remove consecutive duplicates
            unique_coords = []
            for coord in snapped_coords:
                if len(unique_coords) == 0 or coord != unique_coords[-1]:
                    unique_coords.append(coord)
            
            if len(unique_coords) >= 2:
                snapped_line = LineString(unique_coords)
                if snapped_line.is_valid:
                    snapped_geoms.append(snapped_line)
                else:
                    snapped_geoms.append(geom)
            else:
                snapped_geoms.append(geom)
        else:
            snapped_geoms.append(geom)
    
    gdf_proj['geometry'] = snapped_geoms
    gdf_snapped = gdf_proj.to_crs('EPSG:4326')
    
    num_clusters = len(set(cluster_labels))
    print(f"  ✓ Snapped endpoints into {num_clusters} clusters")
    
    return gdf_snapped


def clean_gdf_mv_cables(gdf, merge_proximity_m=50, direction_tolerance_degrees=15, snap_distance_m=15):
    """
    Clean MV cable data using fast distance-based merge approach.
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Input MV cables
    merge_proximity_m : float
        Distance threshold for merging nearby cables (meters)
    direction_tolerance_degrees : float
        Maximum angle difference for cables to be considered parallel
    snap_distance_m : float
        Distance threshold for snapping endpoints (meters)
    """
    print("[424] Cleaning MV Cables (fast distance-based approach)...")
    
    gdf = gdf.copy()
    initial_count = len(gdf)
    
    print(f"  Initial features: {initial_count}")
    
    # Validate geometries
    gdf = gdf[gdf.geometry.is_valid].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    gdf = gdf.reset_index(drop=True)
    print(f"  Valid geometries: {len(gdf)}")
    
    # Extract basic properties
    gdf['tech_type'] = 'transmission_mv'
    gdf['source_layer'] = 424
    gdf['voltage_kv'] = 10.0
    gdf['capacity_mva'] = np.nan
    
    # Step 1: Merge nearby parallel cables (using DBSCAN distance clustering)
    start_time = time.time()
    gdf = merge_nearby_cables_fast(gdf, proximity_threshold_m=merge_proximity_m, 
                                    direction_tolerance_degrees=direction_tolerance_degrees)
    merge_time = time.time() - start_time
    print(f"  Merge step: {merge_time:.1f}s")
    
    gdf = gdf[gdf.geometry.is_valid].copy()
    gdf = gdf.reset_index(drop=True)
    
    # Step 2: Snap endpoints
    start_time = time.time()
    gdf = snap_endpoints_simple(gdf, snap_distance_m=snap_distance_m)
    snap_time = time.time() - start_time
    print(f"  Snap step: {snap_time:.1f}s")
    
    gdf = gdf[gdf.geometry.is_valid].copy()
    gdf = gdf.reset_index(drop=True)
    
    # Calculate final segment properties
    gdf_proj = gdf.to_crs('EPSG:28992')
    gdf['length_m'] = gdf_proj.geometry.length
    
    removed = initial_count - len(gdf)
    print(f"  ✓ Cleaned: {len(gdf)} features remain ({removed} removed/merged)")
    print()
    
    return gdf[['geometry', 'tech_type', 'source_layer', 'capacity_mva', 'voltage_kv', 'length_m']]


# ============================================================================
# EXECUTE CLEANING
# ============================================================================

gdf_mv_cables_clean = None
gdf_hv_stations_clean = None

if 424 in layers_data_1098 and len(layers_data_1098[424]) > 0:
    gdf_mv_cables_clean = clean_gdf_mv_cables(
        layers_data_1098[424],
        merge_proximity_m=50,              # Cables within 50m are candidates
        direction_tolerance_degrees=15,    # Cables within 15° are considered parallel
        snap_distance_m=15                 # Snap endpoints within 15m
    )
else:
    print("[424] No MV cable data available")
    print()

if 641 in layers_data_1098 and len(layers_data_1098[641]) > 0:
    gdf_hv_stations_clean = layers_data_1098[641].copy()
    gdf_hv_stations_clean['tech_type'] = 'source_hv'
    gdf_hv_stations_clean['source_layer'] = 641
    gdf_hv_stations_clean['voltage_kv'] = 30.0
    gdf_hv_stations_clean['lat'] = gdf_hv_stations_clean.geometry.y
    gdf_hv_stations_clean['lon'] = gdf_hv_stations_clean.geometry.x
    print("[641] HV Stations: 1 station (no processing needed)")
    print()
else:
    print("[641] No HV station data available")
    print()

# ============================================================================
# VALIDATION REPORT
# ============================================================================

print("=" * 80)
print("VALIDATION REPORT")
print("=" * 80)

if gdf_mv_cables_clean is not None:
    print(f"\nMV Cables (Layer 424) - AFTER CLEANING:")
    print(f"  Total features: {len(gdf_mv_cables_clean)}")
    print(f"  Total network length: {gdf_mv_cables_clean['length_m'].sum():.1f} m ({gdf_mv_cables_clean['length_m'].sum()/1000:.2f} km)")
    print(f"  Avg segment length: {gdf_mv_cables_clean['length_m'].mean():.1f} m")
    print(f"  Min/Max segment length: {gdf_mv_cables_clean['length_m'].min():.1f} m / {gdf_mv_cables_clean['length_m'].max():.1f} m")

if gdf_hv_stations_clean is not None:
    print(f"\nHV Stations (Layer 641):")
    print(f"  Total stations: {len(gdf_hv_stations_clean)}")

print()
print("✓ Data cleaning complete")
print()

# ============================================================================
# PLOT CLEANED NETWORK (After cleaning)
# ============================================================================

print("=" * 80)
print("PLOTTING CLEANED NETWORK (After Cleaning)")
print("=" * 80)
print()

if gdf_mv_cables_clean is not None and gdf_hv_stations_clean is not None:
    map_cleaned = plot_network_topology(
        gdf_mv_cables=gdf_mv_cables_clean,
        gdf_hv_stations=gdf_hv_stations_clean,
        title="Postal Code 1098 - MV & HV Network (Merged & Snapped)",
        output_file="liander_network_1098_cleaned.html",
        bbox=bbox_1098,
        polygon=neighborhood_polygon,
        zoom_start=15
    )
    print()
else:
    print("⚠ Cannot plot cleaned network - missing cleaned data")
    print()

STAGE 1: DATA CLEANING & VALIDATION (DISTANCE-BASED MERGE → SNAP)

[424] Cleaning MV Cables (fast distance-based approach)...
  Initial features: 993
  Valid geometries: 993
  Merging nearby parallel cables using distance-based clustering...
    Proximity threshold: 50m
    Direction tolerance: 15°
  ✓ Merged 993 cables → 185 trunk(s) (808 cables merged)
  Merge step: 0.4s
  Snapping endpoints (tolerance: 15m)...
  ✓ Snapped endpoints into 233 clusters
  Snap step: 0.0s
  ✓ Cleaned: 185 features remain (808 removed/merged)

[641] HV Stations: 1 station (no processing needed)

VALIDATION REPORT

MV Cables (Layer 424) - AFTER CLEANING:
  Total features: 185
  Total network length: 19340.7 m (19.34 km)
  Avg segment length: 104.5 m
  Min/Max segment length: 0.0 m / 635.3 m

HV Stations (Layer 641):
  Total stations: 1

✓ Data cleaning complete

PLOTTING CLEANED NETWORK (After Cleaning)

Plotting 1 HV stations...
Plotting 185 MV cable segments...
✓ Map saved to: liander_network_1098_cleane

In [149]:
model = calliope.read_yaml('model.yaml')

[2026-03-19 20:20:08] INFO     Math init | loading pre-defined math.
[2026-03-19 20:20:08] INFO     Math init | loading math files {'storage_inter_cluster', 'spores', 'milp', 'base', 'operate'}.
[2026-03-19 20:20:08] INFO     Model: preprocessing data
[2026-03-19 20:20:08] INFO     Math build | building applied math with ['base'].
[2026-03-19 20:20:09] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-19 20:20:09] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2026-03-19 20:20:09] INFO     input data `flow_cap` not defined in model math; it will not be available in the optimisation problem.
[2026-03-19 20:20:09] INFO     input data `storage_cap` not defined in model math; it will not be available in the optimisation problem.
[2026-03-19 20:20:09] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2

In [139]:
model.inputs

<xarray.Dataset> Size: 31kB
Dimensions:                     (costs: 1, techs: 9, nodes: 6, carriers: 1,
                                 timesteps: 48)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 72B 'HV_station_to_data_center...
  * carriers                    (carriers) object 8B 'electricity'
  * nodes                       (nodes) object 48B 'HV_station' ... 'transmis...
  * timesteps                   (timesteps) datetime64[ns] 384B 2024-04-01 .....
Data variables: (12/35)
    cost_interest_rate          (costs) float64 8B 0.1
    bigM                        float64 8B 1e+06
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 72B 'transmission' ... 'transm...
    carrier_in                  (nodes, techs, carriers) bool 54B True ... True
    color                       (techs) object 72B '#823739' ... '#823739'
    ...                          ...
    cost_flow_out               (costs, timesteps, techs) float64 3kB nan ......
    sink_use_equals             (timesteps, techs, nodes) float64 21kB nan .....
    definition_matrix           (nodes, techs, carriers) bool 54B True ... True
    distance                    (techs) float64 72B 0.6481 0.3242 ... nan 0.619
    timestep_resolution         (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0
    timestep_weights            (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0

In [113]:
model.inputs.flow_cap_max.to_series().dropna()  

techs
HV_station_to_data_center_1                  5500.0
HV_station_to_data_center_2                  8800.0
HV_station_to_data_center_3                 13200.0
HV_station_to_transmission_node            100000.0
battery                                     50000.0
pv                                              0.0
supply_grid_power                           50000.0
transmission_node_to_residential_demand    100000.0
Name: flow_cap_max, dtype: float64

In [114]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

techs               nodes             
demand_electricity  data_center_1         240000.000000
                    data_center_2         384000.000000
                    data_center_3         576000.000000
                    residential_demand     31895.209428
Name: sink_use_equals, dtype: float64

In [150]:
model.build(force=True)

[2026-03-19 20:20:13] INFO     Model: backend build starting
[2026-03-19 20:20:14] INFO     Optimisation Model | parameters/lookups | Generated.
[2026-03-19 20:20:14] INFO     Optimisation Model | variables | Generated.
[2026-03-19 20:20:15] INFO     Optimisation Model | global_expressions | Generated.
[2026-03-19 20:20:16] INFO     Optimisation Model | constraints | Generated.
[2026-03-19 20:20:16] INFO     Optimisation Model | piecewise_constraints | Generated.
[2026-03-19 20:20:17] INFO     Optimisation Model | objectives | Generated.
[2026-03-19 20:20:17] INFO     Model: backend build complete


In [116]:
model.backend.parameters

<xarray.Dataset> Size: 30kB
Dimensions:                             (costs: 1, techs: 9, timesteps: 48,
                                         nodes: 6)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 72B 'HV_station_to_dat...
  * timesteps                           (timesteps) datetime64[ns] 384B 2024-...
  * nodes                               (nodes) object 48B 'HV_station' ... '...
Data variables: (12/59)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 72B parameters[...
    ...                                  ...
    storage_cap_per_unit                float64 8B nan
    storage_discharge_depth             float64 8B nan
    storage_initial                     float64 8B nan
    storage_loss                        (techs) object 72B nan nan ... nan nan
    timestep_resolution                 (timesteps) object 384B parameters[ti...
    timestep_weights                    (timesteps) object 384B parameters[ti...

In [151]:
model.solve(solver='gurobi')

[2026-03-19 20:20:19] INFO     Optimisation model | starting model in base mode.
[2026-03-19 20:20:19] INFO     Backend: solver finished running. Time since start of solving optimisation problem: 0:00:00.215125
[2026-03-19 20:20:19] INFO     Postprocessing: applied zero threshold 1e-10 to model results.
[2026-03-19 20:20:19] INFO     Postprocessing: ended. Time since start of solving optimisation problem: 0:00:00.289004
[2026-03-19 20:20:19] INFO     Backend: model solve completed. Time since start of solving optimisation problem: 0:00:00.290317


In [118]:
model.backend.parameters

<xarray.Dataset> Size: 30kB
Dimensions:                             (costs: 1, techs: 9, timesteps: 48,
                                         nodes: 6)
Coordinates:
  * costs                               (costs) object 8B 'monetary'
  * techs                               (techs) object 72B 'HV_station_to_dat...
  * timesteps                           (timesteps) datetime64[ns] 384B 2024-...
  * nodes                               (nodes) object 48B 'HV_station' ... '...
Data variables: (12/59)
    area_use_max                        float64 8B nan
    area_use_min                        float64 8B nan
    area_use_per_flow_cap               float64 8B nan
    available_area                      float64 8B nan
    bigM                                object 8B parameters[bigM][0]
    cost_flow_cap_per_distance          (costs, techs) object 72B parameters[...
    ...                                  ...
    storage_cap_per_unit                float64 8B nan
    storage_discharge_depth             float64 8B nan
    storage_initial                     float64 8B nan
    storage_loss                        (techs) object 72B nan nan ... nan nan
    timestep_resolution                 (timesteps) object 384B parameters[ti...
    timestep_weights                    (timesteps) object 384B parameters[ti...

In [143]:
model.results

<xarray.Dataset> Size: 190kB
Dimensions:                     (nodes: 6, techs: 9, carriers: 1,
                                 timesteps: 48, costs: 1)
Coordinates:
  * techs                       (techs) object 72B 'HV_station_to_data_center...
  * nodes                       (nodes) object 48B 'HV_station' ... 'transmis...
  * carriers                    (carriers) object 8B 'electricity'
  * timesteps                   (timesteps) datetime64[ns] 384B 2024-04-01 .....
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/22)
    flow_cap                    (nodes, techs, carriers) float64 432B 5.033e+...
    link_flow_cap               (techs) float64 72B 5.033e+03 ... 1.054e+03
    flow_out                    (nodes, techs, carriers, timesteps) float64 21kB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 21kB ...
    flow_export                 (nodes, techs, carriers, timesteps) float64 21kB ...
    source_use                  (nodes, techs, timesteps) float64 21kB nan .....
    ...                          ...
    min_cost_optimisation       float64 8B 1.023e+07
    capacity_factor             (nodes, techs, carriers, timesteps) float64 21kB ...
    systemwide_capacity_factor  (techs, carriers) float64 72B 0.4909 ... 0.3416
    systemwide_levelised_cost   (techs, costs, carriers) float64 72B 8.851 .....
    total_levelised_cost        (costs, carriers) float64 8B 8.225
    unmet_sum                   float64 8B nan

In [144]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes       techs                            costs   
HV_station  HV_station_to_data_center_1      monetary    1.049572e+06
            HV_station_to_data_center_2      monetary    9.045106e+05
            HV_station_to_data_center_3      monetary    2.263507e+05
            HV_station_to_transmission_node  monetary    3.763036e+05
            supply_grid_power                monetary    4.698720e+06
Name: cost, dtype: float64

In [121]:
lcoes = (
    model.results.systemwide_levelised_cost.sel(carriers="electricity")
    .to_series()
    .dropna()
)
lcoes.head()

techs                            costs   
HV_station_to_data_center_1      monetary     8.746431
HV_station_to_data_center_2      monetary     4.617631
HV_station_to_data_center_3      monetary     0.755363
HV_station_to_transmission_node  monetary    21.646333
battery                          monetary     0.000021
Name: systemwide_levelised_cost, dtype: float64

In [122]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

In [152]:
df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

                         techs           timesteps  Flow in/out (kWh)
0  HV_station_to_data_center_1 2024-04-01 00:00:00         -32.672577
1  HV_station_to_data_center_1 2024-04-01 01:00:00         -32.672577
2  HV_station_to_data_center_1 2024-04-01 02:00:00         -32.672577
3  HV_station_to_data_center_1 2024-04-01 03:00:00         -32.672577
4  HV_station_to_data_center_1 2024-04-01 04:00:00         -32.672577


In [153]:
carriers = ["electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

        nodes                        techs     carriers           timesteps  \
0  HV_station  HV_station_to_data_center_1  electricity 2024-04-01 00:00:00   
1  HV_station  HV_station_to_data_center_1  electricity 2024-04-01 01:00:00   
2  HV_station  HV_station_to_data_center_1  electricity 2024-04-01 02:00:00   
3  HV_station  HV_station_to_data_center_1  electricity 2024-04-01 03:00:00   
4  HV_station  HV_station_to_data_center_1  electricity 2024-04-01 04:00:00   

   Flow in/out (kWh)  
0       -5032.672577  
1       -5032.672577  
2       -5032.672577  
3       -5032.672577  
4       -5032.672577  


In [147]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

           nodes              techs     carriers  Flow capacity (kW)
0     HV_station  supply_grid_power  electricity        27968.681023
1  data_center_1            battery  electricity          993.540858
2  data_center_1                 pv  electricity         1000.000000
3  data_center_2            battery  electricity         9057.469159
4  data_center_2                 pv  electricity         1000.000000


In [154]:
with open("model.yaml", "r", encoding="utf-8") as f:
    model_def = yaml.safe_load(f)

node_techs = {
    node: list((node_data.get("techs") or {}).keys())
    for node, node_data in model_def.get("nodes", {}).items()
}

In [155]:
# Build a simple system map (nodes + links)
nodes = pd.read_csv("nodes_coordinates.csv")
links = pd.read_csv("links_techs.csv")

flow_cap = (
    model.results.flow_cap.to_series().dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
flow_cap_lookup = dict(zip(flow_cap["techs"], flow_cap["Flow capacity (kW)"]))

center = [nodes.latitude.mean(), nodes.longitude.mean()]
system_map = folium.Map(location=center, zoom_start=15, tiles="CartoDB voyager")

# Add link lines
for _, row in links.iterrows():
    from_row = nodes.loc[nodes.nodes == row["link_from"]].iloc[0]
    to_row = nodes.loc[nodes.nodes == row["link_to"]].iloc[0]
    capacity = flow_cap_lookup.get(row["techs"], row.get("flow_cap_max"))
    popup = (
        f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}"
    )
    if capacity is not None:
        popup += f"<br>Capacity: {capacity:.2f} kW"
    folium.PolyLine(
        locations=[[from_row.latitude, from_row.longitude], [to_row.latitude, to_row.longitude]],
        color=row.get("color", "#1f77b4"),
        weight=3,
        opacity=1,
        popup=popup,
    ).add_to(system_map)

# Add node markers
color_map = model.inputs.color.to_series().to_dict()
for _, row in nodes.iterrows():
    node = row["nodes"]
    techs = node_techs.get(node, [])
    base_types = (
        model.inputs.base_tech.sel(techs=techs).to_series().to_dict()
        if techs
        else {}
    )

    node_type = "Other"
    if any(t == "demand" for t in base_types.values()):
        node_type = "Demand"
    elif any(t == "supply" for t in base_types.values()):
        node_type = "Supply"

    marker_color = "#666666"
    for tech in techs:
        if tech in color_map:
            marker_color = color_map[tech]
            break

    popup = f"<b>{node}</b> ({node_type})<br>Techs: {', '.join(techs)}"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=marker_color,
        fill=True,
        fillColor=marker_color,
        fillOpacity=1,
        popup=popup,
    ).add_to(system_map)

system_map.save("system_map.html")

In [ ]:
output_folder = os.path.join(os.getcwd(), "outputs")
os.makedirs(output_folder, exist_ok=True)
output_csv = os.path.join(output_folder, "bill_of_materials.csv")

# 1) Flow capacities per node/tech (kW)
cap_s = (
    model.results.flow_cap.to_series()
    .dropna()
    .where(lambda x: x != 0)
    .dropna()
)
cap_df = cap_s.to_frame("capacity_kw").reset_index()

# 2) Collapse carriers (if present) so we get exactly one row per (node, tech)
if {"nodes", "techs"}.issubset(cap_df.columns):
    group_cols = ["nodes", "techs"]
else:
    raise ValueError(f"Unexpected flow_cap index columns: {cap_df.columns.tolist()}")

if "carriers" in cap_df.columns:
    cap_node_tech = (
        cap_df.groupby(group_cols, as_index=False, dropna=False)
        .agg(
            capacity_kw=("capacity_kw", "sum"),
            carriers=("carriers", lambda s: ",".join(sorted({str(x) for x in s.dropna()}))),
        )
    )
else:
    cap_node_tech = (
        cap_df.groupby(group_cols, as_index=False, dropna=False)
        .agg(capacity_kw=("capacity_kw", "sum"))
    )

# 3) Tech metadata (unique by tech)
tech_meta = model.inputs.base_tech.to_series().rename("base_tech").reset_index()

if "name" in model.inputs:
    tech_meta = tech_meta.merge(
        model.inputs.name.to_series().rename("name").reset_index(),
        how="left",
        on="techs",
    )
else:
    tech_meta["name"] = pd.NA

if "distance" in model.inputs:
    tech_meta = tech_meta.merge(
        model.inputs.distance.to_series().rename("distance_km").reset_index(),
        how="left",
        on="techs",
    )
else:
    tech_meta["distance_km"] = pd.NA

tech_meta["distance_m"] = tech_meta["distance_km"] * 1000.0
tech_meta["name"] = tech_meta["name"].fillna(tech_meta["techs"])

# 4) Join + filter to links + supply + storage
df = cap_node_tech.merge(
    tech_meta[["techs", "name", "base_tech", "distance_m"]],
    how="left",
    on="techs",
)

allowed_base_tech = {"transmission", "supply", "storage"}
df = df[df["base_tech"].fillna("").str.lower().isin(allowed_base_tech)].copy()

# 5) Export (keeps nodes + techs, no cross-node aggregation)
sort_cols = [c for c in ["nodes", "base_tech", "name", "techs"] if c in df.columns]
df = df.sort_values(by=sort_cols).reset_index(drop=True)

df.to_csv(output_csv, index=False)

NameError: name 'os' is not defined